# AIPerf Use Case 2 — Auditing Raw Per-Request Exports

Every AIPerf run exports **every individual request** to `profile_export.jsonl`. This notebook loads that raw export and computes statistics AIPerf's summary table doesn't ship — e.g. a P75, when the report only has P50/P90/P99.

These use-case notebooks demonstrate AIPerf capabilities directly. This is the foundation of the repo's convention that **the raw AIPerf export is the deliverable** — no processed report layer.

**What this notebook is — and is not.** UC2 is the *single-request microscope*: one row per request, so you can compute any latency distribution, percentile, tail, or per-request scaling that describes **what one user experiences**. It deliberately does **not** report system-wide throughput, request-rate, or goodput — those are aggregate, concurrency-dependent numbers that per-request records structurally can't produce (they depend on how many requests overlapped in wall-clock time). Those live in the sibling notebooks: **UC1** (run-totals throughput + concurrency sweep), **UC4** (goodput), **UC6** (throughput paired with GPU telemetry). Read UC2's latency detail as the UX picture, not a capacity verdict.

## Table of contents

1. [Overview](#1-overview)
2. [Requirements](#2-requirements)
3. [Input — anatomy of `profile_export.jsonl`](#3-input--anatomy-of-profile_exportjsonl)
4. [Results analysis](#4-results-analysis)

## 1. Overview

The summary table (CSV/JSON) gives avg/min/max/p50/p90/p99/std per metric. The JSONL gives the underlying per-request records, so you can:

- compute **any percentile or custom statistic** (P75, trimmed means, ...);
- **slice** by input/output length, worker, or time window;
- build your own dashboards — the format is open, nothing is locked to AIPerf's presentation.

## 2. Requirements

- `aiperf` CLI installed and on `PATH` (from the NGC AIPerf image, or `pip install aiperf`). Pin whatever version you use (repo convention: record the AIPerf version per run) — this notebook was written against `release/0.3.0`.
- A reachable OpenAI-compatible endpoint (NIM, vLLM, TGI, ...) serving the model under test.
- Notebook Python deps: `pip install -r notebooks/requirements.txt` (jupyter, pandas, matplotlib).
- Tip: AIPerf's live dashboard is designed for a terminal. If it renders poorly inside Jupyter, check `aiperf profile --help` for a plain/simple UI option in your version.
- **Data**: an existing AIPerf artifact directory. Point `ARTIFACT_DIR` (below) at any prior run — e.g. this repo's scenario outputs, or the Use Case 1 notebook's `notebooks/artifacts/aiperf-uc1-synthetic`.

In [ ]:
# Relative to REPO_ROOT — any prior run's --artifact-dir (kept under notebooks/).
ARTIFACT_DIR = "notebooks/artifacts/aiperf-uc1-synthetic"
print(f"Artifact dir: {ARTIFACT_DIR}")

In [ ]:
import json
import os
import shutil
import subprocess
import urllib.request
from pathlib import Path

# Notebook is expected to run from notebooks/ inside the repo (Jupyter's default cwd).
REPO_ROOT = Path.cwd().parent if not (Path.cwd() / "model-selection").exists() else Path.cwd()
assert (REPO_ROOT / "model-selection").exists(), (
    f"Could not find the repo root from {Path.cwd()} — run this notebook from the notebooks/ directory."
)
print(f"Repo root: {REPO_ROOT}")

aiperf_path = shutil.which("aiperf")
if aiperf_path is None:
    print("aiperf CLI not found on PATH.")
else:
    print(f"aiperf found at: {aiperf_path}")
    version = subprocess.run(["aiperf", "--version"], capture_output=True, text=True)
    print((version.stdout or version.stderr).strip())


## 3. Input — anatomy of `profile_export.jsonl`

One JSON object per line, one line per request. Abridged example record:

```json
{
  "metadata": {
    "session_num": 87,
    "x_request_id": "abd8df1a-7904-4aa0-8107-0d74ba0ac0d7",
    "turn_index": 0,
    "request_start_ns": 1763066701865462000,
    "request_end_ns": 1763066703082535666,
    "worker_id": "worker_b431129c"
  },
  "metrics": {
    "time_to_first_token": {"value": 582.66, "unit": "ms"},
    "request_latency": {"value": 1210.008, "unit": "ms"},
    "inter_token_latency": {"value": 3.25, "unit": "ms"},
    "input_sequence_length": {"value": 1000, "unit": "tokens"},
    "output_sequence_length": {"value": 194, "unit": "tokens"}
  }
}
```

Send 10,000 requests and you get 10,000 of these. Field names/nesting can vary slightly across AIPerf versions — the loader below flattens whatever `metrics` contains.

## 4. Results analysis

Load the JSONL into a flat DataFrame (one row per request, one column per metric), then compute custom statistics.

In [ ]:
import pandas as pd

artifact_dir = REPO_ROOT / ARTIFACT_DIR
jsonl_path = next(artifact_dir.rglob("profile_export.jsonl"), None)
assert jsonl_path is not None, f"No profile_export.jsonl under {artifact_dir}."

records = []
with open(jsonl_path) as f:
    for line in f:
        rec = json.loads(line)
        row = {k: v for k, v in rec.get("metadata", {}).items()}
        for name, m in rec.get("metrics", {}).items():
            row[name] = m["value"] if isinstance(m, dict) and "value" in m else m
        records.append(row)

df = pd.DataFrame(records)
print(f"{len(df)} requests, columns: {sorted(df.columns)}")

# First 5 raw per-request records, flattened to one row per request — metadata
# columns (session/request id, worker, timestamps) plus one column per metric.
df.head()


In [ ]:
# A motivating example: percentiles the summary table doesn't include.
ttft = df["time_to_first_token"].dropna()

percentiles = [25, 50, 75, 90, 99]
custom = pd.DataFrame({
    "percentile": [f"P{p}" for p in percentiles],
    "ttft_ms": [ttft.quantile(p / 100) for p in percentiles],
})
print(f"Total requests analyzed: {len(ttft)}")

# Custom TTFT percentiles (incl. P25/P75) computed directly from the raw
# per-request data — not present in AIPerf's summary table (which only ships P50/P90/P99).
custom


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ttft.plot.hist(bins=40, ax=axes[0])
axes[0].set_title("TTFT distribution")
axes[0].set_xlabel("ms")

axes[1].plot(ttft.sort_values().values, ttft.rank(pct=True).sort_values().values)
axes[1].set_title("TTFT ECDF (read off any percentile)")
axes[1].set_xlabel("ms")
axes[1].set_ylabel("fraction of requests")

if {"input_sequence_length", "request_latency"} <= set(df.columns):
    axes[2].scatter(df["input_sequence_length"], df["request_latency"], s=8, alpha=0.4)
    axes[2].set_title("Request latency vs. input length")
    axes[2].set_xlabel("ISL (tokens)")
    axes[2].set_ylabel("latency (ms)")

plt.tight_layout()


In [ ]:
# Decode-side view: the ITL half of the TTFT/ITL story. TTFT covers the first
# token (prefill); everything after it is decode, summarized by inter-token
# latency. The summary table ships ITL percentiles the same way it does TTFT —
# but here we can also derive per-request quantities it never reports.
if {"inter_token_latency", "output_sequence_length"} <= set(df.columns):
    itl = df["inter_token_latency"].dropna()

    percentiles = [25, 50, 75, 90, 99]
    itl_custom = pd.DataFrame({
        "percentile": [f"P{p}" for p in percentiles],
        "itl_ms": [itl.quantile(p / 100) for p in percentiles],
    })

    # Per-request derived metrics (computed inline, not stored back on df):
    #   output tokens/sec  = 1000 / ITL_ms   (per-stream decode rate)
    #   decode time (ms)   = request_latency - TTFT   (time generating after token 1)
    out_tok_s = (1000.0 / itl).replace([float("inf")], pd.NA).dropna()
    decode_ms = None
    if {"request_latency", "time_to_first_token"} <= set(df.columns):
        decode_ms = (df["request_latency"] - df["time_to_first_token"]).dropna()

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    itl.plot.hist(bins=40, ax=axes[0])
    axes[0].set_title("Inter-token latency (decode) distribution")
    axes[0].set_xlabel("ms per output token")
    out_tok_s.plot.hist(bins=40, ax=axes[1])
    axes[1].set_title("Per-request output tokens/sec (= 1000 / ITL)")
    axes[1].set_xlabel("tokens/sec per stream")
    plt.tight_layout()

    # ITL percentiles (incl. P25/P75) + the median per-stream decode rate, and a
    # check that the two phases tile the whole request: TTFT + decode ~= latency.
    print(f"Median ITL: {itl.median():.2f} ms  ->  ~{1000 / itl.median():,.0f} output tokens/sec per stream")
    if decode_ms is not None:
        print(f"Median TTFT (prefill): {df['time_to_first_token'].median():.0f} ms  +  "
              f"median decode: {decode_ms.median():.0f} ms  ~=  "
              f"median request latency: {df['request_latency'].median():.0f} ms")
    display(itl_custom)
else:
    print("inter_token_latency / output_sequence_length not in this export — skipping the decode-side view.")

In [ ]:
# Prefill vs. decode scaling: which phase dominates, and how each grows with
# sequence length. Prefill cost (TTFT) should rise with input length (ISL);
# decode cost (ITL) is largely flat vs. output length (OSL) unless something
# degrades as generations get longer. The per-request records make both visible.
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

if {"input_sequence_length", "time_to_first_token"} <= set(df.columns):
    sub = df[["input_sequence_length", "time_to_first_token"]].dropna()
    axes[0].scatter(sub["input_sequence_length"], sub["time_to_first_token"], s=8, alpha=0.4)
    axes[0].set_title("Prefill: TTFT vs. input length")
    axes[0].set_xlabel("ISL (tokens)")
    axes[0].set_ylabel("TTFT (ms)")
    if len(sub) >= 2 and sub["input_sequence_length"].nunique() >= 2:
        slope = sub["time_to_first_token"].cov(sub["input_sequence_length"]) / sub["input_sequence_length"].var()
        print(f"Prefill: TTFT rises ~{slope * 100:.1f} ms per +100 input tokens "
              f"(corr {sub['time_to_first_token'].corr(sub['input_sequence_length']):+.2f}).")
else:
    axes[0].set_title("TTFT vs. ISL — fields not in this export")

if {"output_sequence_length", "inter_token_latency"} <= set(df.columns):
    sub = df[["output_sequence_length", "inter_token_latency"]].dropna()
    axes[1].scatter(sub["output_sequence_length"], sub["inter_token_latency"], s=8, alpha=0.4)
    axes[1].set_title("Decode: ITL vs. output length")
    axes[1].set_xlabel("OSL (tokens)")
    axes[1].set_ylabel("ITL (ms/token)")
    if len(sub) >= 2 and sub["output_sequence_length"].nunique() >= 2:
        corr = sub["inter_token_latency"].corr(sub["output_sequence_length"])
        print(f"Decode: ITL vs. OSL correlation {corr:+.2f} "
              f"(near 0 = decode rate stable across output length; strongly positive = it slows on long generations).")
else:
    axes[1].set_title("ITL vs. OSL — fields not in this export")

plt.tight_layout()


In [ ]:
# Custom time-series over the run, from raw per-request timestamps. This is the
# post-hoc, any-bucket-size equivalent of UC5's --slice-duration: UC5 asks AIPerf
# to bucket into fixed windows *at run time*; here we re-derive arbitrary windows
# from request_start_ns on data already collected (UC5's own Notes cell points
# here — "the per-request JSONL can reproduce any custom bucketing").
if "request_start_ns" in df.columns:
    BUCKET_S = 10  # window size in seconds — change freely, no re-run needed
    ts = df.dropna(subset=["request_start_ns"]).copy()
    ts["t_s"] = (ts["request_start_ns"] - ts["request_start_ns"].min()) / 1e9
    ts["bucket"] = (ts["t_s"] // BUCKET_S) * BUCKET_S

    metric_cols = [c for c in ["time_to_first_token", "inter_token_latency"] if c in ts.columns]
    by_bucket = ts.groupby("bucket")[metric_cols].median()

    fig, ax = plt.subplots(figsize=(9, 4))
    for c in metric_cols:
        ax.plot(by_bucket.index, by_bucket[c], marker="o", label=f"median {c}")
    ax.set_title(f"Latency over the run ({BUCKET_S}s windows) — warm-up tail / drift")
    ax.set_xlabel("seconds since first request")
    ax.set_ylabel("ms")
    ax.legend()
    plt.tight_layout()
    print(f"{len(by_bucket)} windows of {BUCKET_S}s. A high first window that settles = warm-up; "
          "a steady upward creep at constant load = degradation (leak / KV-cache fragmentation).")

    # Load-generator balance: each client worker should carry a similar share of
    # requests at similar latency. A skewed count or a slow worker is a client-side
    # artifact, not the endpoint's fault.
    if "worker_id" in ts.columns:
        by_worker = ts.groupby("worker_id").agg(
            requests=("worker_id", "size"),
            median_ttft_ms=("time_to_first_token", "median"),
        )
        print("\nPer-worker load balance (client-side load generator):")
        display(by_worker)
else:
    print("request_start_ns not in this export — skipping the time-series / per-worker view.")


### Notes

- The decode-side, scaling, and time-series cells above all read the **same** per-request JSONL — no extra files. TTFT (prefill) and ITL (decode) together tile the whole request latency; the scaling view shows which phase grows with sequence length; the time-series view re-buckets by `request_start_ns` into any window (the post-hoc equivalent of UC5's run-time `--slice-duration`), and the per-`worker_id` table flags client-side load-generator imbalance.
- For **aggregate** numbers — system-wide output-token throughput, request rate, goodput — the per-request records aren't enough (they depend on concurrency overlap). Those come from the summary CSV's run-totals section (UC1's `read_export()`), from UC4 (goodput), or from UC6 (throughput + GPU telemetry).
- This post-hoc workflow is exactly how this repo computes goodput today (`--goodput` isn't passed in the scenario scripts) — see the Use Case 4 notebook.